# LangChain: Memory

## Outline

- ConversationBufferMemory
- ConversationBufferWindowMemory
- ConversationTokenBufferMemory
- ConversationSummaryMemory


In [9]:
from enum import StrEnum

from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory import ChatMessageHistory
from pydantic import SecretStr


class LMStudio(StrEnum):
    BASE_URL = "http://localhost:1234/v1"
    API_KEY = "lm-studio"
    MODEL = "model-identifier"


# create a new client
llm = ChatOpenAI(
    api_key=SecretStr(LMStudio.API_KEY),
    model=LMStudio.MODEL,
    base_url=LMStudio.BASE_URL,
    temperature=0,
)

# prompt + llm + parser = runnable/chain

chain = llm

# create a memory session

store = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    """
    Retrieves or creates a ChatMessageHistory object for a given session ID.
    """
    if session_id not in store:
        print(f"Creating new history for session: {session_id}")
        store[session_id] = ChatMessageHistory()
    else:
        print(f"Using existing history for session: {session_id}")
    return store[session_id]


# create a conversation chain
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    # history_messages_key="history",  # matches memory_key="history" above
)
config = {"session_id": "session_id_1"}

response1 = conversation.invoke("My name is Bob.", config=config)
response2 = conversation.invoke("What's my name?", config=config)

response2


Creating new history for session: session_id_1
Using existing history for session: session_id_1


AIMessage(content='Your name is Bob! You just told me that. 😄\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 50, 'total_tokens': 63, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'gemma-3-12b-it', 'system_fingerprint': 'gemma-3-12b-it', 'id': 'chatcmpl-vqpcqsb9jcofnfxktuopdq', 'finish_reason': 'stop', 'logprobs': None}, id='run-7639e018-7553-4165-98b0-30efca927c2a-0', usage_metadata={'input_tokens': 50, 'output_tokens': 13, 'total_tokens': 63, 'input_token_details': {}, 'output_token_details': {}})